# Burst 3 (~65 CUs on NVIDIA A100 40GB / L4 24GB): Week 3 / Prototype 2 — MoE Router MSAN Profiling, Physical Expert Surgery & 1-Epoch `TRL` Distillation Recovery

**Sprint**: Applied AI Foundry (v4.0) — **Prototype 2 of the 3-Prototype Startup Tournament**  
**Target HF Repo**: [`garg-achal/olmoe-1b-7b-finops-pruned-week3`](https://huggingface.co/garg-achal/olmoe-1b-7b-finops-pruned-week3)  
**Venture Context**: [`curons.org`](https://curons.org) — Running 6.9B MoE Models inside Single Consumer/Edge GPUs

### Architectural Objective (Why MoE Surgery Saves Enterprise GPU Spend)
1. **The MoE Memory Paradox**: `OLMoE-1B-7B` activates only `8` of `64` experts per token (`1.3B` active FLOPs), **yet all `64` experts (`6.9B` params $\approx 13.8\text{ GB}$ BF16) must stay pinned in GPU HBM**, starving `vLLM` PagedAttention KV-cache slots.
2. **Gate-Weighted MSAN beats Frequency**: Low-frequency experts can still emit critical syntactic pivots (JSON braces, SQL joins). We score every expert $(l, e)$ across 2,000 enterprise FinOps/K8s/SQL tokens using both Routing Frequency and **Mean Squared Activation Norm (MSAN)**:
   $$\text{MSAN}_{l,e} = \frac{1}{T} \sum_{t=1}^T g_{l,e}(\mathbf{h}_t)^2 \cdot \|\text{Expert}_{l,e}(\mathbf{h}_t)\|_2^2$$
3. **Physical Tensor Surgery + Distillation**: Physically excise `20` experts per layer (`64 -> 44`, freeing ~31% static VRAM), update `config.num_experts = 44`, and heal router softmax boundaries with a 1-epoch `TRL` distillation pass from the unpruned 6.9B teacher.

> **Colab Runtime**: Select **Runtime → Change runtime type → A100 GPU** (recommended for fastest 2,000-prompt profiling + distillation, or **L4 GPU**), verify `HF_TOKEN` in Colab Secrets, and click **Runtime → Run all**.

In [ ]:
%pip install -q -U "transformers>=5.0.0" "trl>=0.15.0" "datasets>=3.0.0" "accelerate>=1.2.0" "huggingface_hub>=0.28.0" "safetensors>=0.4.5"

import torch

assert torch.cuda.is_available(), (
    "CUDA GPU not detected! Go to Runtime -> Change runtime type and select A100 or L4 GPU."
)

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
compute_dtype = torch.bfloat16 if major >= 8 else torch.float16

print(f"GPU Detected       : {gpu_name} (SM {major}.{minor})")
print(f"Total GPU VRAM     : {vram_gb:.2f} GB")
print(f"Selected Precision : {compute_dtype}")

In [ ]:
import os
from huggingface_hub import login, notebook_login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Authenticated to Hugging Face Hub via Colab Secret HF_TOKEN.")
else:
    print("HF_TOKEN secret not found. Launching interactive notebook login...")
    notebook_login()

In [ ]:
# ==============================================================================
# PART 1: LOAD OLMOE-1B-7B & GENERATE 2,000 ENTERPRISE DOMAIN PROFILING PROMPTS
# ==============================================================================
import json
import math
from pathlib import Path
import random
import time

from datasets import Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "allenai/OLMoE-1B-7B-0924-Instruct"
HUB_REPO_ID = "garg-achal/olmoe-1b-7b-finops-pruned-week3"
OUT_DIR = Path("outputs/olmoe-1b-7b-finops-pruned-week3")
OUT_DIR.mkdir(parents=True, exist_ok=True)
K_KEEP_EXPERTS = 44  # Prune 64 -> 44 experts/layer (-31.25% experts)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

torch.cuda.reset_peak_memory_stats()
t_load = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=compute_dtype,
    device_map="auto",
)
model.eval()

num_layers = model.config.num_hidden_layers
orig_num_experts = model.config.num_experts
top_k = model.config.num_experts_per_tok
orig_params = sum(p.numel() for p in model.parameters())
baseline_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

print(
    f"Loaded {MODEL_ID} in {time.perf_counter()-t_load:.1f}s | Layers={num_layers} | "
    f"Experts/Layer={orig_num_experts} (top-{top_k}) | Params={orig_params:,} | Peak VRAM={baseline_vram_gb:.2f} GB"
)


def build_domain_corpus(n_prompts: int = 2000, seed: int = 42) -> tuple[list[str], list[str]]:
    """Build 2,000 enterprise FinOps / Kubernetes / SQL profiling prompts + 40 held-out eval prompts."""
    rng = random.Random(seed)
    workloads = ["Kubernetes PodCrashLoopBackOff", "vLLM PagedAttention KV-cache", "PostgreSQL index scan",
                 "AWS g5.2xlarge A10G inference", "FastAPI async batch gateway", "PyTorch forward hook probe",
                 "Snowflake warehouse concurrency", "CUDA OOM batch fragmentation"]
    models = ["Qwen3-4B", "Qwen2.5-7B", "OLMoE-1B-7B", "Llama-3.1-8B", "Gemma-3-4B"]
    metrics = ["P95 TTFT < 200ms", "VRAM utilization > 88%", "inter-token latency < 18ms",
               "blended cost < $0.25/1M tokens", "99.5% JSON schema compliance"]
    actions = ["quantize to W4A16 Marlin", "enable FP8 KV-cache compression", "enforce padding_side='left'",
               "consolidate via vLLM --enable-lora", "prune inactive MoE experts via MSAN"]

    prompts = []
    for i in range(n_prompts + 40):
        w = workloads[i % len(workloads)]
        m = models[(i // 3) % len(models)]
        met = metrics[(i // 5) % len(metrics)]
        act = actions[(i // 7) % len(actions)]
        ctx = [2048, 4096, 8192][i % 3]
        conc = [16, 32, 64, 128][i % 4]
        prompt = (
            f"Incident #{i:04d}: Diagnose {w} for {m} serving {conc} concurrent {ctx}-token sessions. "
            f"Target SLA: {met}. Explain why we should {act} and estimate JSON telemetry."
        )
        prompts.append(prompt)
    rng.shuffle(prompts)
    return prompts[:n_prompts], prompts[n_prompts:]


def format_prompt(text: str) -> str:
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": text}],
            add_generation_prompt=True,
            tokenize=False,
        )
    return f"User: {text}\nAssistant:"


profile_prompts, eval_prompts = build_domain_corpus(n_prompts=2000)
print(f"Corpus ready: {len(profile_prompts)} profiling prompts | {len(eval_prompts)} held-out eval prompts.")


def compute_perplexity(eval_model: AutoModelForCausalLM, texts: list[str], batch_size: int = 8) -> float:
    eval_model.eval()
    total_nll = 0.0
    total_tokens = 0
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            batch = [format_prompt(t) for t in texts[start : start + batch_size]]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=160, return_tensors="pt").to(eval_model.device)
            labels = enc["input_ids"].clone()
            labels[enc["attention_mask"] == 0] = -100
            out = eval_model(**enc, labels=labels, use_cache=False)
            valid_toks = int((labels[:, 1:] != -100).sum().item())
            total_nll += float(out.loss.item()) * max(valid_toks, 1)
            total_tokens += max(valid_toks, 1)
    return math.exp(total_nll / max(total_tokens, 1))


baseline_ppl = compute_perplexity(model, eval_prompts)
print(f"Baseline Unpruned (64 experts) Eval Perplexity: {baseline_ppl:.4f}")

In [ ]:
# ==============================================================================
# PART 2: COLLECT TEACHER DISTILLATION TARGETS & PROFILE ROUTER MSAN (2,000 PROMPTS)
# ==============================================================================
# First generate 120 crisp teacher completions from the unpruned 6.9B model for 1-epoch distillation healing
distill_prompts = profile_prompts[:120]
teacher_examples = []

print("Generating 120 unpruned 6.9B teacher targets for post-pruning TRL distillation...")
t_teach = time.perf_counter()
with torch.inference_mode():
    for start in range(0, len(distill_prompts), 12):
        batch_p = distill_prompts[start : start + 12]
        rendered = [format_prompt(p) for p in batch_p]
        enc = tokenizer(rendered, padding=True, truncation=True, max_length=128, return_tensors="pt").to(model.device)
        gen = model.generate(
            **enc,
            max_new_tokens=96,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
        prompt_len = enc["input_ids"].shape[1]
        for j, p_txt in enumerate(batch_p):
            ans = tokenizer.decode(gen[j, prompt_len:], skip_special_tokens=True).strip()
            if getattr(tokenizer, "chat_template", None):
                full_text = tokenizer.apply_chat_template(
                    [{"role": "user", "content": p_txt}, {"role": "assistant", "content": ans}],
                    tokenize=False,
                )
            else:
                full_text = f"User: {p_txt}\nAssistant: {ans}"
            teacher_examples.append({"text": full_text})

print(f"Generated {len(teacher_examples)} teacher distillation samples in {time.perf_counter()-t_teach:.1f}s")

# Now attach MoE Profiling Hooks across all 16 layers to compute Routing Frequency & MSAN
route_counts = torch.zeros(num_layers, orig_num_experts, dtype=torch.float64)
msan_sums = torch.zeros(num_layers, orig_num_experts, dtype=torch.float64)
total_tokens_per_layer = torch.zeros(num_layers, dtype=torch.float64)
norm_topk = bool(getattr(model.config, "norm_topk_prob", False))


def _eval_expert_and_msan(mlp_mod: nn.Module, x_flat: torch.Tensor, topk_idx: torch.Tensor, topk_w: torch.Tensor, l_idx: int) -> None:
    experts_obj = mlp_mod.experts
    is_modlist = isinstance(experts_obj, nn.ModuleList)

    for e_idx in range(orig_num_experts):
        mask = topk_idx == e_idx
        if not mask.any():
            continue
        tok_pos, k_pos = mask.nonzero(as_tuple=True)
        route_counts[l_idx, e_idx] += float(tok_pos.numel())

        x_sub = x_flat[tok_pos]
        g_sub = topk_w[tok_pos, k_pos].float()

        if is_modlist:
            e_out = experts_obj[e_idx](x_sub)
        else:
            # Handle transformers v5 fused 3D parameter OlmoeExperts layout cleanly
            if hasattr(experts_obj, "gate_up_proj"):
                gu_w = experts_obj.gate_up_proj[e_idx]
                dn_w = experts_obj.down_proj[e_idx]
                h_dim = x_sub.shape[-1]
                gu = F.linear(x_sub, gu_w) if gu_w.shape[1] == h_dim else x_sub @ gu_w
                gate, up = gu.chunk(2, dim=-1)
                act = F.silu(gate) * up
                e_out = F.linear(act, dn_w) if dn_w.shape[1] == act.shape[-1] else act @ dn_w
            else:
                e_out = x_sub

        sq_norm = e_out.float().pow(2).sum(dim=-1)
        msan_sums[l_idx, e_idx] += float(((g_sub ** 2) * sq_norm).sum().item())


def make_moe_profiler_hook(l_idx: int):
    def _hook(module, inputs, output):
        hs = inputs[0]
        x_flat = hs.reshape(-1, hs.shape[-1])
        total_tokens_per_layer[l_idx] += float(x_flat.shape[0])
        logits = module.gate(x_flat).float()
        probs = torch.softmax(logits, dim=-1)
        topk_w, topk_idx = torch.topk(probs, k=top_k, dim=-1)
        if norm_topk:
            topk_w = topk_w / (topk_w.sum(dim=-1, keepdim=True) + 1e-12)
        _eval_expert_and_msan(module, x_flat, topk_idx, topk_w, l_idx)
    return _hook


prof_handles = [
    model.model.layers[l].mlp.register_forward_hook(make_moe_profiler_hook(l))
    for l in range(num_layers)
]

print("Profiling 2,000 domain prompts across all 16 OLMoE layers (Frequency + MSAN)...")
t_prof = time.perf_counter()
try:
    with torch.inference_mode():
        for start in range(0, len(profile_prompts), 20):
            batch_txt = [format_prompt(p) for p in profile_prompts[start : start + 20]]
            enc = tokenizer(batch_txt, padding=True, truncation=True, max_length=128, return_tensors="pt").to(model.device)
            _ = model(**enc, use_cache=False)
            if (start // 20) % 25 == 0:
                print(f"  Processed {min(start + 20, len(profile_prompts))}/2000 prompts...")
finally:
    for h in prof_handles:
        h.remove()

torch.cuda.synchronize()
prof_time_s = time.perf_counter() - t_prof
freq_pct = (route_counts / total_tokens_per_layer.unsqueeze(1).clamp(min=1.0)) * 100.0
msan_scores = msan_sums / total_tokens_per_layer.unsqueeze(1).clamp(min=1.0)
print(f"Router & MSAN profiling finished in {prof_time_s:.1f}s (Tokens/layer={int(total_tokens_per_layer[0].item()):,})")

In [ ]:
# ==============================================================================
# PART 3: PHYSICAL EXPERT TENSOR SURGERY (64 -> 44 EXPERTS) & ZERO-SHOT CHECK
# ==============================================================================
pruning_summary = []

for l in range(num_layers):
    layer_freq = freq_pct[l]
    layer_msan = msan_scores[l]

    # Composite Importance Score: 50% Normalized Routing Frequency + 50% Normalized MSAN
    norm_f = layer_freq / (layer_freq.max() + 1e-12)
    norm_m = layer_msan / (layer_msan.max() + 1e-12)
    composite = 0.5 * norm_f + 0.5 * norm_m

    keep_idx = torch.topk(composite, k=K_KEEP_EXPERTS).indices.sort().values
    drop_idx = [i for i in range(orig_num_experts) if i not in set(keep_idx.tolist())]

    mlp_mod = model.model.layers[l].mlp
    gate_dev = mlp_mod.gate.weight.device
    keep_dev = keep_idx.to(device=gate_dev)

    # 1. Slice Router Gate matrix [64, 2048] -> [44, 2048]
    mlp_mod.gate.weight = nn.Parameter(mlp_mod.gate.weight.data[keep_dev].clone())
    mlp_mod.gate.out_features = K_KEEP_EXPERTS
    if hasattr(mlp_mod, "num_experts"):
        mlp_mod.num_experts = K_KEEP_EXPERTS

    # 2. Slice physical Expert weight tensors
    if isinstance(mlp_mod.experts, nn.ModuleList):
        mlp_mod.experts = nn.ModuleList([mlp_mod.experts[i] for i in keep_idx.tolist()])
    else:
        for p_name, param in list(mlp_mod.experts.named_parameters(recurse=False)):
            if param.shape[0] == orig_num_experts:
                setattr(mlp_mod.experts, p_name, nn.Parameter(param.data[keep_idx.to(param.device)].clone()))
        if hasattr(mlp_mod.experts, "num_experts"):
            mlp_mod.experts.num_experts = K_KEEP_EXPERTS

    pruning_summary.append({
        "layer": l,
        "kept_experts": K_KEEP_EXPERTS,
        "pruned_experts": len(drop_idx),
        "retained_msan_pct": round(float(layer_msan[keep_idx].sum() / (layer_msan.sum() + 1e-12)) * 100.0, 2),
        "mean_pruned_freq_pct": round(float(layer_freq[drop_idx].mean().item()), 3),
    })

model.config.num_experts = K_KEEP_EXPERTS
pruned_params = sum(p.numel() for p in model.parameters())
param_reduction_pct = (1.0 - pruned_params / orig_params) * 100.0

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
zeroshot_pruned_ppl = compute_perplexity(model, eval_prompts)

print(
    f"Physical Surgery Complete: {orig_params:,} -> {pruned_params:,} params (-{param_reduction_pct:.2f}%)"
)
print(
    f"Eval Perplexity: Baseline={baseline_ppl:.4f} -> Zero-Shot Pruned={zeroshot_pruned_ppl:.4f}"
)

In [ ]:
# ==============================================================================
# PART 4: 1-EPOCH TRL DISTILLATION RECOVERY & RELOAD BENCHMARK
# ==============================================================================
import gc
import inspect
from trl import SFTConfig, SFTTrainer

# Freeze heavy expert FFN matrices and unfreeze Router Gates + Attention + Norms (~270M params)
# Avoids 57 GB full-param AdamW OOM so 1-epoch healing runs in ~13.5 GB VRAM on both L4 & A100!
for p_name, param in model.named_parameters():
    if any(k in p_name for k in (".mlp.gate.", "self_attn", "layernorm", "norm")):
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable_heal = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Distillation Healing Params (Router Gates + Attn + Norms): {trainable_heal:,} / {pruned_params:,}")

model.config.use_cache = False
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

distill_ds = Dataset.from_list(teacher_examples)
sft_params = inspect.signature(SFTConfig.__init__).parameters
sft_kwargs = dict(
    output_dir="outputs/olmoe_distill_tmp",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_steps=2,
    logging_steps=3,
    save_strategy="no",
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    report_to="none",
    seed=42,
)
if "dataset_text_field" in sft_params:
    sft_kwargs["dataset_text_field"] = "text"
if "max_length" in sft_params:
    sft_kwargs["max_length"] = 256
elif "max_seq_length" in sft_params:
    sft_kwargs["max_seq_length"] = 256

model.train()
trainer = SFTTrainer(
    model=model,
    train_dataset=distill_ds,
    args=SFTConfig(**sft_kwargs),
    processing_class=tokenizer,
)

t_distill = time.perf_counter()
train_res = trainer.train()
torch.cuda.synchronize()
distill_time_s = time.perf_counter() - t_distill

model.gradient_checkpointing_disable()
model.config.use_cache = True
for param in model.parameters():
    param.requires_grad = False

healed_ppl = compute_perplexity(model, eval_prompts)
ppl_delta_pct = ((healed_ppl - baseline_ppl) / max(baseline_ppl, 1e-6)) * 100.0

model.save_pretrained(OUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUT_DIR)
disk_gb = sum(f.stat().st_size for f in OUT_DIR.rglob("*.safetensors")) / (1024 ** 3)

print(
    f"1-Epoch Distillation Complete in {distill_time_s:.1f}s | "
    f"Healed Eval Perplexity: {healed_ppl:.4f} (vs Baseline {baseline_ppl:.4f}, delta={ppl_delta_pct:+.2f}%)"
)

# Reload from disk cleanly to verify Hugging Face / vLLM loadability & VRAM reduction
del trainer, model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

reloaded = AutoModelForCausalLM.from_pretrained(OUT_DIR, dtype=compute_dtype, device_map="auto")
reloaded.eval()
pruned_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
vram_saved_pct = (1.0 - pruned_vram_gb / max(baseline_vram_gb, 1e-6)) * 100.0

test_enc = tokenizer(format_prompt(eval_prompts[0]), return_tensors="pt").to(reloaded.device)
torch.cuda.synchronize()
t_gen = time.perf_counter()
with torch.inference_mode():
    out_toks = reloaded.generate(**test_enc, max_new_tokens=96, do_sample=False, pad_token_id=tokenizer.pad_token_id)
torch.cuda.synchronize()
gen_s = time.perf_counter() - t_gen
tok_s = (out_toks.shape[-1] - test_enc["input_ids"].shape[-1]) / max(gen_s, 1e-6)
sample_gen = tokenizer.decode(out_toks[0, test_enc["input_ids"].shape[-1]:], skip_special_tokens=True)

print(
    f"Reloaded Pruned MoE VRAM: {pruned_vram_gb:.2f} GB (vs {baseline_vram_gb:.2f} GB baseline, -{vram_saved_pct:.1f}%) "
    f"| Gen Speed: {tok_s:.1f} tok/s | Disk: {disk_gb:.2f} GB"
)

In [ ]:
# ==============================================================================
# PART 5: SAVE TELEMETRY + MODEL CARD & PUBLISH PRUNED MOE CHECKPOINT TO HF HUB
# ==============================================================================
from huggingface_hub import HfApi

telemetry = {
    "base_model": MODEL_ID,
    "hub_repo_id": HUB_REPO_ID,
    "gpu_name": gpu_name,
    "orig_experts_per_layer": orig_num_experts,
    "kept_experts_per_layer": K_KEEP_EXPERTS,
    "orig_params": orig_params,
    "pruned_params": pruned_params,
    "param_reduction_pct": round(param_reduction_pct, 2),
    "baseline_vram_gb": round(baseline_vram_gb, 3),
    "pruned_vram_gb": round(pruned_vram_gb, 3),
    "vram_saved_pct": round(vram_saved_pct, 2),
    "disk_gb": round(disk_gb, 3),
    "baseline_ppl": round(baseline_ppl, 4),
    "zeroshot_pruned_ppl": round(zeroshot_pruned_ppl, 4),
    "healed_ppl": round(healed_ppl, 4),
    "ppl_delta_pct": round(ppl_delta_pct, 2),
    "profile_time_s": round(prof_time_s, 1),
    "distill_time_s": round(distill_time_s, 1),
    "eval_tok_per_sec": round(tok_s, 2),
    "layer_pruning_summary": pruning_summary,
}

with open(OUT_DIR / "router_msan_telemetry.json", "w", encoding="utf-8") as f:
    json.dump(telemetry, f, indent=2)

layer_rows = "\n".join(
    f"| Layer {r['layer']:02d} | {orig_num_experts} -> {r['kept_experts']} | "
    f"{r['retained_msan_pct']:.2f}% | {r['mean_pruned_freq_pct']:.3f}% |"
    for r in pruning_summary
)

readme = f"""---
base_model: {MODEL_ID}
tags:
  - moe
  - expert-pruning
  - msan
  - olmoe
  - knowledge-distillation
  - trl
  - finops
  - curons
  - applied-ai-foundry
license: apache-2.0
---

# {HUB_REPO_ID.split('/')[-1]}: Structurally Pruned MoE (`64 -> {K_KEEP_EXPERTS}` Experts/Layer) via MSAN & `TRL` Distillation

Built as **Prototype 2 (Week 3 / Burst 3)** of the **Applied AI Foundry (`v4.0`)** Sprint (**[`curons.org`](https://curons.org)** | **[`@garg-achal`](https://huggingface.co/garg-achal)**).

Physically excises `20` low-MSAN/low-frequency non-domain experts per layer from `{MODEL_ID}` across 2,000 FinOps/K8s/SQL prompts, then recovers routing boundaries with a 1-epoch `TRL` distillation pass from the unpruned 6.9B teacher.

## 1. Surgery & Recovery Scorecard (`{gpu_name}`)

| Metric | Unpruned (`64` experts) | Zero-Shot Pruned (`{K_KEEP_EXPERTS}` experts) | Healed (`1-Epoch TRL Distill`) |
| :--- | :--- | :--- | :--- |
| **Total Parameters** | `{orig_params:,}` | `{pruned_params:,}` (`-{param_reduction_pct:.2f}%`) | `{pruned_params:,}` (`-{param_reduction_pct:.2f}%`) |
| **Model VRAM Footprint** | `{baseline_vram_gb:.2f} GB` | — | **`{pruned_vram_gb:.2f} GB` (`-{vram_saved_pct:.1f}%`)** |
| **Held-Out Eval Perplexity** | `{baseline_ppl:.4f}` | `{zeroshot_pruned_ppl:.4f}` | **`{healed_ppl:.4f}` (`{ppl_delta_pct:+.2f}%`)** |
| **Eval Speed** | — | — | `{tok_s:.1f} tok/s` |

## 2. Layer-by-Layer MSAN Retention Telemetry

| Layer | Experts (`Before -> After`) | Retained MSAN Energy | Mean Pruned Expert Routing Freq |
| :--- | :--- | :--- | :--- |
{layer_rows}
"""

with open(OUT_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme)

api = HfApi()
api.create_repo(repo_id=HUB_REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path=str(OUT_DIR),
    repo_id=HUB_REPO_ID,
    repo_type="model",
    commit_message="feat: publish Burst 3 MSAN-pruned OLMoE-1B-7B checkpoint & telemetry",
)

print(f"SUCCESS! Pruned MoE checkpoint published to https://huggingface.co/{HUB_REPO_ID}")
print("\n--- COPY TELEMETRY JSON BELOW ---")
print(json.dumps({k: v for k, v in telemetry.items() if k != "layer_pruning_summary"}, indent=2))
print("ACTION REQUIRED: Click 'Runtime -> Disconnect and delete runtime' NOW to stop Colab CU burn.")